# Teacher-Student ResNet50 120k Notebook

This notebook runs teacher-student distillation with a ResNet50 teacher on the larger labeled `120k / 10k / 20k` wafer split.

Updated workflow:
- load the dedicated `120k` TS-ResNet50 config
- inspect the larger labeled dataset split and CUDA status
- optionally train the model and save artifacts under `artifacts/x64/ts_resnet50_120k/`
- run the default shared evaluation for apples-to-apples comparison
- optionally launch a same-notebook ablation sweep over teacher layers and `topk_ratio`
- run a post-training teacher-student score sweep over branch weights and wafer-level reductions
- compare the default score against stronger rescoring variants in the same notebook



In [1]:
from pathlib import Path
import json
import subprocess
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics
from wafer_defect.models.ts_distillation import build_ts_distillation_from_config
from wafer_defect.scoring import spatial_max, spatial_mean, topk_spatial_mean



In [2]:
CONFIG_OPTIONS = {
    "ts_resnet50_120k": REPO_ROOT / "configs" / "training" / "train_ts_resnet50_120k.toml",
}
CONFIG_NAME = "ts_resnet50_120k"
CONFIG_PATH = CONFIG_OPTIONS[CONFIG_NAME]
THRESHOLD_QUANTILE = 0.95
RUN_TRAINING = True
RUN_DEFAULT_EVALUATION = True

RUN_ABLATION_SWEEP = False
ABLATION_PROFILE = "focused"  # "focused" or "full"
if ABLATION_PROFILE == "focused":
    ABLATION_LAYERS = ["layer1", "layer2"]
    ABLATION_TOPK_RATIOS = [0.15, 0.20, 0.25]
    ABLATION_EPOCHS_OVERRIDE = 15
else:
    ABLATION_LAYERS = ["layer1", "layer2", "layer3"]
    ABLATION_TOPK_RATIOS = [0.10, 0.15, 0.20, 0.25, 0.30]
    ABLATION_EPOCHS_OVERRIDE = 20
ABLATION_REUSE_EXISTING = True

SCORE_SWEEP_WEIGHTS = [
    (1.0, 1.0),
    (1.0, 0.0),
    (0.0, 1.0),
    (2.0, 1.0),
    (1.0, 2.0),
    (1.0, 0.5),
    (0.5, 1.0),
]
SCORE_SWEEP_REDUCTIONS = [
    ("mean", None),
    ("max", None),
    ("topk_mean", 0.01),
    ("topk_mean", 0.05),
    ("topk_mean", 0.10),
    ("topk_mean", 0.20),
]

config = load_toml(CONFIG_PATH)
output_dir = REPO_ROOT / config["run"]["output_dir"]
evaluation_dir = output_dir / "evaluation"

print(f"Using config: {CONFIG_NAME} -> {CONFIG_PATH.name}")
print(f"Artifact dir: {output_dir}")
config




Using config: ts_resnet50_120k -> train_ts_resnet50_120k.toml
Artifact dir: C:\Users\User\Desktop\Term 8\Deep Learning\Project\DeepLearning-Group8\artifacts\x64\ts_resnet50_120k


{'run': {'output_dir': 'artifacts/x64/ts_resnet50_120k', 'seed': 42},
 'data': {'metadata_csv': 'data/processed/x64/wm811k_patchcore_custom/metadata_train120000_a6000_val10000_a500_test20000_a1000.csv',
  'image_size': 64,
  'batch_size': 16,
  'num_workers': 0},
 'training': {'epochs': 30,
  'learning_rate': 0.0003,
  'weight_decay': 1e-05,
  'device': 'auto',
  'early_stopping_patience': 5,
  'early_stopping_min_delta': 0.0001,
  'checkpoint_every': 5,
  'resume_from': ''},
 'model': {'type': 'ts_distillation',
  'teacher_backbone': 'resnet50',
  'teacher_layer': 'layer2',
  'teacher_pretrained': True,
  'teacher_input_size': 224,
  'normalize_teacher_input': True,
  'feature_autoencoder_hidden_dim': 128,
  'student_weight': 1.0,
  'autoencoder_weight': 1.0,
  'score_student_weight': 1.0,
  'score_autoencoder_weight': 0.0,
  'reduction': 'topk_mean',
  'topk_ratio': 0.2}}

In [3]:
requested_device = str(config["training"].get("device", "auto"))
if requested_device == "auto":
    resolved_device_name = "cuda" if torch.cuda.is_available() else "cpu"
else:
    resolved_device_name = requested_device

device = torch.device(resolved_device_name)

cuda_summary = {
    "config_device": requested_device,
    "torch_cuda_available": torch.cuda.is_available(),
    "resolved_device": resolved_device_name,
    "cuda_device_count": torch.cuda.device_count(),
}
if torch.cuda.is_available():
    cuda_summary["cuda_device_name"] = torch.cuda.get_device_name(0)

cuda_summary



{'config_device': 'auto',
 'torch_cuda_available': True,
 'resolved_device': 'cuda',
 'cuda_device_count': 1,
 'cuda_device_name': 'NVIDIA GeForce RTX 3070 Ti Laptop GPU'}

In [4]:
metadata_path = REPO_ROOT / config["data"]["metadata_csv"]
dataset_prep_config = REPO_ROOT / "configs" / "data" / "data_patchcore_wrn50_120k.toml"

if metadata_path.exists():
    print(f"Using existing dataset metadata: {metadata_path}")
else:
    print(f"Missing dataset metadata: {metadata_path}")
    prep_command = [
        sys.executable,
        "-u",
        str(REPO_ROOT / "scripts" / "prepare_wm811k.py"),
        "--config",
        str(dataset_prep_config),
    ]
    print("Running:", " ".join(prep_command))
    process = subprocess.Popen(
        prep_command,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, prep_command)
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Dataset prep finished but metadata still does not exist: {metadata_path}"
        )
    print(f"Prepared dataset metadata: {metadata_path}")



Missing dataset metadata: C:\Users\User\Desktop\Term 8\Deep Learning\Project\DeepLearning-Group8\data\processed\x64\wm811k_patchcore_custom\metadata_train120000_a6000_val10000_a500_test20000_a1000.csv
Running: c:\Users\User\Desktop\Term 8\Deep Learning\Project\DeepLearning-Group8\venv\Scripts\python.exe -u C:\Users\User\Desktop\Term 8\Deep Learning\Project\DeepLearning-Group8\scripts\prepare_wm811k.py --config C:\Users\User\Desktop\Term 8\Deep Learning\Project\DeepLearning-Group8\configs\data\data_patchcore_wrn50_120k.toml
C:\Users\User\Desktop\Term 8\Deep Learning\Project\DeepLearning-Group8\src\wafer_defect\data\legacy_pickle.py:15: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(handle, encoding="latin1")
Using arrays directory: data\processed\x64\wm811k_patchcore_custom\arrays_train120000_a6000_val10000_a500_test20000_a1000
Sav

In [5]:
image_size = int(config["data"].get("image_size", 64))
batch_size = int(config["data"].get("batch_size", 64))
num_workers = int(config["data"].get("num_workers", 0))
metadata = pd.read_csv(metadata_path)

display(metadata.head())
display(metadata.groupby(["split", "is_anomaly"]).size().rename("count").reset_index())

train_dataset = WaferMapDataset(metadata_path, split="train", image_size=image_size)
val_dataset = WaferMapDataset(metadata_path, split="val", image_size=image_size)
test_dataset = WaferMapDataset(metadata_path, split="test", image_size=image_size)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")




,array_path,label,defect_type,is_anomaly,split,source_split,original_height,original_width
0,data/processed/x64/wm811k_patchcore_custom/arr...,none,none,0,train,Test,29,26
1,data/processed/x64/wm811k_patchcore_custom/arr...,none,none,0,train,Test,33,29
2,data/processed/x64/wm811k_patchcore_custom/arr...,none,none,0,train,Test,39,50
3,data/processed/x64/wm811k_patchcore_custom/arr...,none,none,0,train,Test,25,27
4,data/processed/x64/wm811k_patchcore_custom/arr...,none,none,0,train,Training,26,26


,split,is_anomaly,count
0,test,0,19000
1,test,1,1000
2,train,0,114000
3,train,1,6000
4,val,0,9500
5,val,1,500


train=120000, val=10000, test=20000


In [ ]:
if RUN_TRAINING:
    train_command = [
        sys.executable,
        "-u",
        str(REPO_ROOT / "scripts" / "train_ts_distillation.py"),
        "--config",
        str(CONFIG_PATH),
    ]
    print("Running:", " ".join(train_command))
    process = subprocess.Popen(
        train_command,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, train_command)
else:
    print(f"Skipping training. Using existing checkpoint at {best_model_path}")
    if not best_model_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {best_model_path}. Set RUN_TRAINING = True first.")



Running: c:\Users\User\Desktop\Term 8\Deep Learning\Project\DeepLearning-Group8\venv\Scripts\python.exe -u C:\Users\User\Desktop\Term 8\Deep Learning\Project\DeepLearning-Group8\scripts\train_ts_distillation.py --config C:\Users\User\Desktop\Term 8\Deep Learning\Project\DeepLearning-Group8\configs\training\train_ts_resnet50_120k.toml


In [ ]:
history_df = pd.read_json(output_dir / "history.json")
training_summary = json.loads((output_dir / "summary.json").read_text(encoding="utf-8"))

display(history_df.tail())
training_summary



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val")
axes[0].set_title("Teacher-Student Total Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_distillation_loss"], marker="o", label="train distillation")
axes[1].plot(history_df["epoch"], history_df["val_distillation_loss"], marker="o", label="val distillation")
axes[1].plot(history_df["epoch"], history_df["train_autoencoder_loss"], marker="o", linestyle="--", label="train feature AE")
axes[1].plot(history_df["epoch"], history_df["val_autoencoder_loss"], marker="o", linestyle="--", label="val feature AE")
axes[1].set_title("Teacher-Student Branch Losses")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()



In [ ]:
if RUN_DEFAULT_EVALUATION:
    evaluate_command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "evaluate_reconstruction_model.py"),
        "--checkpoint",
        str(best_model_path),
        "--config",
        str(CONFIG_PATH),
        "--model-type",
        "ts_distillation",
        "--threshold-quantile",
        str(THRESHOLD_QUANTILE),
        "--output-dir",
        str(evaluation_dir),
    ]
    print("Running:", " ".join(evaluate_command))
    subprocess.run(evaluate_command, cwd=REPO_ROOT, check=True)

default_summary_path = evaluation_dir / "summary.json"
if not default_summary_path.exists():
    raise FileNotFoundError(f"Evaluation summary not found: {default_summary_path}")

default_eval_summary = json.loads(default_summary_path.read_text(encoding="utf-8"))
display(pd.Series(default_eval_summary["metrics_at_validation_threshold"]))
display(pd.Series(default_eval_summary["best_threshold_sweep"]))



## Optional TS Ablation Sweep

This section can run a compact retraining sweep over teacher feature layers and deployment `topk_ratio` values from the same notebook. The default `focused` preset tests only the highest-value variants first, while `full` restores the broader grid. It writes temporary configs, trains each variant with the standard script, evaluates it with the shared evaluator, and saves a sortable summary table.


In [ ]:
from copy import deepcopy


def format_toml_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, int) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, float):
        return repr(value)
    return json.dumps(str(value))


def write_simple_toml(path, config_dict):
    lines = []
    for section_name, section_values in config_dict.items():
        lines.append(f"[{section_name}]")
        for key, value in section_values.items():
            lines.append(f"{key} = {format_toml_value(value)}")
        lines.append("")
    path.write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")


def stream_command(command, cwd):
    print("Running:", " ".join(str(part) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def build_ablation_variants(base_config, layers, topk_ratios, epochs_override=None):
    variants = []
    backbone_name = str(base_config["model"]["teacher_backbone"]).lower()
    image_size_name = f"x{int(base_config['data'].get('image_size', 64))}"
    for teacher_layer in layers:
        for topk_ratio in topk_ratios:
            variant_config = deepcopy(base_config)
            variant_name = f"ts_{backbone_name}_{teacher_layer}_topk{int(round(topk_ratio * 100)):02d}"
            variant_config["run"]["output_dir"] = f"artifacts/{image_size_name}/{variant_name}"
            variant_config["model"]["teacher_layer"] = teacher_layer
            variant_config["model"]["topk_ratio"] = float(topk_ratio)
            variant_config["model"]["score_student_weight"] = 1.0
            variant_config["model"]["score_autoencoder_weight"] = 0.0
            variant_config["training"]["resume_from"] = ""
            if epochs_override is not None:
                variant_config["training"]["epochs"] = int(epochs_override)
            variants.append({
                "name": variant_name,
                "teacher_layer": teacher_layer,
                "topk_ratio": float(topk_ratio),
                "config": variant_config,
            })
    return variants


def run_ablation_variant(variant, threshold_quantile, reuse_existing):
    variant_name = variant["name"]
    variant_config = variant["config"]
    generated_config_dir = REPO_ROOT / "artifacts" / "generated_configs"
    generated_config_dir.mkdir(parents=True, exist_ok=True)
    config_path = generated_config_dir / f"{variant_name}.toml"
    write_simple_toml(config_path, variant_config)

    variant_output_dir = REPO_ROOT / variant_config["run"]["output_dir"]
    variant_output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = variant_output_dir / "best_model.pt"
    evaluation_output_dir = variant_output_dir / "evaluation"
    evaluation_output_dir.mkdir(parents=True, exist_ok=True)
    summary_path = evaluation_output_dir / "summary.json"

    if reuse_existing and checkpoint_path.exists():
        print(f"Reusing checkpoint for {variant_name}: {checkpoint_path}")
    else:
        stream_command(
            [
                sys.executable,
                "-u",
                REPO_ROOT / "scripts" / "train_ts_distillation.py",
                "--config",
                config_path,
            ],
            cwd=REPO_ROOT,
        )

    if reuse_existing and summary_path.exists():
        print(f"Reusing evaluation for {variant_name}: {summary_path}")
    else:
        stream_command(
            [
                sys.executable,
                REPO_ROOT / "scripts" / "evaluate_reconstruction_model.py",
                "--checkpoint",
                checkpoint_path,
                "--config",
                config_path,
                "--model-type",
                "ts_distillation",
                "--threshold-quantile",
                str(threshold_quantile),
                "--output-dir",
                evaluation_output_dir,
            ],
            cwd=REPO_ROOT,
        )

    train_summary = json.loads((variant_output_dir / "summary.json").read_text(encoding="utf-8"))
    evaluation_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    metrics = evaluation_summary["metrics_at_validation_threshold"]
    best_sweep = evaluation_summary["best_threshold_sweep"]
    return {
        "name": variant_name,
        "teacher_backbone": variant_config["model"]["teacher_backbone"],
        "teacher_layer": variant["teacher_layer"],
        "topk_ratio": variant["topk_ratio"],
        "epochs": int(variant_config["training"]["epochs"]),
        "best_epoch": int(train_summary["best_epoch"]),
        "best_val_loss": float(train_summary["best_val_loss"]),
        "precision": float(metrics["precision"]),
        "recall": float(metrics["recall"]),
        "f1": float(metrics["f1"]),
        "auroc": float(metrics["auroc"]),
        "auprc": float(metrics["auprc"]),
        "best_sweep_f1": float(best_sweep["f1"]),
        "threshold": float(metrics["threshold"]),
        "output_dir": str(variant_output_dir),
        "config_path": str(config_path),
    }


ablation_variants = build_ablation_variants(
    base_config=config,
    layers=ABLATION_LAYERS,
    topk_ratios=ABLATION_TOPK_RATIOS,
    epochs_override=ABLATION_EPOCHS_OVERRIDE,
)

pd.DataFrame(
    [{
        "name": variant["name"],
        "teacher_layer": variant["teacher_layer"],
        "topk_ratio": variant["topk_ratio"],
        "epochs": variant["config"]["training"]["epochs"],
    } for variant in ablation_variants]
)



In [ ]:
ablation_summary_path = evaluation_dir / "ablation_sweep_summary.csv"

if RUN_ABLATION_SWEEP:
    ablation_rows = []
    for variant in ablation_variants:
        print(f"\n=== Ablation: {variant['name']} ===")
        ablation_rows.append(
            run_ablation_variant(
                variant=variant,
                threshold_quantile=THRESHOLD_QUANTILE,
                reuse_existing=ABLATION_REUSE_EXISTING,
            )
        )
    ablation_df = pd.DataFrame(ablation_rows).sort_values(["f1", "auroc"], ascending=False).reset_index(drop=True)
    ablation_df.to_csv(ablation_summary_path, index=False)
    print(f"Saved ablation sweep summary to {ablation_summary_path}")
elif ablation_summary_path.exists():
    ablation_df = pd.read_csv(ablation_summary_path)
    print(f"Loaded existing ablation summary from {ablation_summary_path}")
else:
    ablation_df = pd.DataFrame()
    print("Skipping ablation sweep. Set RUN_ABLATION_SWEEP = True to launch it.")

ablation_df.head(10)



In [ ]:
if ablation_df.empty:
    print("No ablation sweep results available yet.")
else:
    display(ablation_df)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    top_rows = ablation_df.head(8)

    axes[0].bar(top_rows["name"], top_rows["f1"], color="#2a9d8f")
    axes[0].set_title("Top Ablations by F1")
    axes[0].set_ylabel("F1")
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].bar(top_rows["name"], top_rows["auroc"], color="#457b9d")
    axes[1].set_title("Top Ablations by AUROC")
    axes[1].set_ylabel("AUROC")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()



## Teacher-Student Score Sweep

The first teacher-student result looked more like a calibration problem than a training-collapse problem. This sweep keeps the trained checkpoint fixed and tests whether different branch weights and wafer-level reductions produce a stronger operating point under the same validation-threshold rule.


In [ ]:
checkpoint = torch.load(best_model_path, map_location="cpu")
score_sweep_model = build_ts_distillation_from_config(config, image_size=image_size)
score_sweep_model.load_state_dict(checkpoint["model_state_dict"])
score_sweep_model.to(device)
score_sweep_model.eval()

def collect_normalized_maps(model, dataloader, device):
    student_maps = []
    auto_maps = []
    labels = []
    with torch.inference_mode():
        for inputs, batch_labels in dataloader:
            inputs = inputs.to(device)
            student_map, auto_map = model.raw_anomaly_maps(inputs)
            student_maps.append((student_map / model.student_map_scale.clamp_min(1e-6)).cpu())
            auto_maps.append((auto_map / model.autoencoder_map_scale.clamp_min(1e-6)).cpu())
            labels.append(batch_labels.cpu())
    return torch.cat(student_maps, dim=0), torch.cat(auto_maps, dim=0), torch.cat(labels, dim=0).numpy()

val_student_maps, val_auto_maps, val_labels = collect_normalized_maps(score_sweep_model, val_loader, device)
test_student_maps, test_auto_maps, test_labels = collect_normalized_maps(score_sweep_model, test_loader, device)

print(tuple(val_student_maps.shape), tuple(test_student_maps.shape))



In [ ]:
def reduce_anomaly_map(anomaly_map, reduction, topk_ratio):
    if reduction == "mean":
        return spatial_mean(anomaly_map)
    if reduction == "max":
        return spatial_max(anomaly_map)
    return topk_spatial_mean(anomaly_map, topk_ratio=topk_ratio)

score_sweep_rows = []
for student_weight, auto_weight in SCORE_SWEEP_WEIGHTS:
    val_map = student_weight * val_student_maps + auto_weight * val_auto_maps
    test_map = student_weight * test_student_maps + auto_weight * test_auto_maps
    for reduction, topk_ratio in SCORE_SWEEP_REDUCTIONS:
        val_scores = reduce_anomaly_map(val_map, reduction, topk_ratio).numpy()
        test_scores = reduce_anomaly_map(test_map, reduction, topk_ratio).numpy()

        threshold = float(pd.Series(val_scores[val_labels == 0]).quantile(THRESHOLD_QUANTILE))
        metrics = summarize_threshold_metrics(test_labels, test_scores, threshold)
        _, best_sweep = sweep_threshold_metrics(test_labels, test_scores)

        score_sweep_rows.append(
            {
                "name": f"s{student_weight:g}_a{auto_weight:g}_{reduction}" + ("" if topk_ratio is None else f"_r{topk_ratio:.2f}"),
                "student_weight": student_weight,
                "auto_weight": auto_weight,
                "reduction": reduction,
                "topk_ratio": np.nan if topk_ratio is None else topk_ratio,
                "threshold": threshold,
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "auroc": metrics["auroc"],
                "auprc": metrics["auprc"],
                "predicted_anomalies": metrics["predicted_anomalies"],
                "best_sweep_f1": best_sweep["f1"],
            }
        )

score_sweep_df = pd.DataFrame(score_sweep_rows).sort_values(["f1", "auprc", "auroc"], ascending=False).reset_index(drop=True)
score_sweep_path = evaluation_dir / "score_sweep_summary.csv"
score_sweep_df.to_csv(score_sweep_path, index=False)

display(score_sweep_df.head(15))
print(f"Saved score sweep summary to {score_sweep_path}")



In [ ]:
default_row = pd.DataFrame(
    [
        {
            "name": "default_config_score",
            "precision": default_eval_summary["metrics_at_validation_threshold"]["precision"],
            "recall": default_eval_summary["metrics_at_validation_threshold"]["recall"],
            "f1": default_eval_summary["metrics_at_validation_threshold"]["f1"],
            "auroc": default_eval_summary["metrics_at_validation_threshold"]["auroc"],
            "auprc": default_eval_summary["metrics_at_validation_threshold"]["auprc"],
            "best_sweep_f1": default_eval_summary["best_threshold_sweep"]["f1"],
        }
    ]
)

selected_row = score_sweep_df.iloc[0].copy()
selected_row_df = pd.DataFrame([selected_row])
comparison_df = pd.concat([default_row, selected_row_df[["name", "precision", "recall", "f1", "auroc", "auprc", "best_sweep_f1"]]], ignore_index=True)
display(comparison_df)

selected_variant_summary = selected_row.to_dict()
selected_variant_summary_path = evaluation_dir / "selected_score_variant.json"
selected_variant_summary_path.write_text(json.dumps(selected_variant_summary, indent=2), encoding="utf-8")
selected_variant_summary



In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path.cwd()
if "evaluation_dir" in globals():
    local_evaluation_dir = Path(evaluation_dir)
else:
    candidate_dirs = [
        repo_root / "artifacts" / "x64" / "ts_resnet50_120k" / "evaluation",
        repo_root / "artifacts" / "x64" / "ts_resnet50_120k" / "evaluation",
    ]
    existing_dirs = [path for path in candidate_dirs if path.exists()]
    if not existing_dirs:
        raise FileNotFoundError("Could not find a TS-Res18 evaluation directory.")
    local_evaluation_dir = existing_dirs[0]

default_summary_path = local_evaluation_dir / "summary.json"
selected_variant_path = local_evaluation_dir / "selected_score_variant.json"
if not default_summary_path.exists():
    raise FileNotFoundError(f"Evaluation summary not found: {default_summary_path}")
if not selected_variant_path.exists():
    raise FileNotFoundError(f"Selected score variant not found: {selected_variant_path}")

default_eval_summary = json.loads(default_summary_path.read_text(encoding="utf-8"))
selected_variant = json.loads(selected_variant_path.read_text(encoding="utf-8"))

default_row = pd.DataFrame(
    [
        {
            "name": "default_config_score",
            "precision": default_eval_summary["metrics_at_validation_threshold"]["precision"],
            "recall": default_eval_summary["metrics_at_validation_threshold"]["recall"],
            "f1": default_eval_summary["metrics_at_validation_threshold"]["f1"],
            "auroc": default_eval_summary["metrics_at_validation_threshold"]["auroc"],
            "auprc": default_eval_summary["metrics_at_validation_threshold"]["auprc"],
            "best_sweep_f1": default_eval_summary["best_threshold_sweep"]["f1"],
        }
    ]
)
selected_row_df = pd.DataFrame([selected_variant])
comparison_df = pd.concat(
    [
        default_row,
        selected_row_df[["name", "precision", "recall", "f1", "auroc", "auprc", "best_sweep_f1"]],
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["default", "best sweep variant"], comparison_df["f1"], color=["#8d99ae", "#2a9d8f"])
axes[0].set_title("Validation-Threshold F1")
axes[0].set_ylabel("F1")

axes[1].bar(["default", "best sweep variant"], comparison_df["auroc"], color=["#8d99ae", "#2a9d8f"])
axes[1].set_title("AUROC")
axes[1].set_ylabel("AUROC")

plt.tight_layout()
plt.show()



## Defect-Type Breakdown

This cell recomputes the defect-type recall for the **selected deployed score** and saves it beside the evaluation outputs.


In [ ]:
# DEFECT_BREAKDOWN_CELL
metadata_path = REPO_ROOT / config["data"]["metadata_csv"]
test_metadata = pd.read_csv(metadata_path)
test_metadata = test_metadata[test_metadata["split"] == "test"].reset_index(drop=True)

selected_scores_df = pd.read_csv(local_evaluation_dir / "test_scores.csv").reset_index(drop=True)
if len(selected_scores_df) != len(test_metadata):
    raise ValueError(f"Length mismatch: scores={len(selected_scores_df)} metadata={len(test_metadata)}")

selected_threshold = float(selected_variant["threshold"])
analysis_df = test_metadata.copy()
analysis_df["score"] = selected_scores_df["score"]
analysis_df["predicted_anomaly"] = (analysis_df["score"] > selected_threshold).astype(int)

defect_breakdown_df = (
    analysis_df.loc[analysis_df["is_anomaly"] == 1]
    .groupby("defect_type")
    .agg(
        count=("defect_type", "size"),
        detected=("predicted_anomaly", "sum"),
        mean_score=("score", "mean"),
        median_score=("score", "median"),
    )
    .reset_index()
)
defect_breakdown_df["detected"] = defect_breakdown_df["detected"].astype(int)
defect_breakdown_df["missed"] = defect_breakdown_df["count"] - defect_breakdown_df["detected"]
defect_breakdown_df["recall"] = defect_breakdown_df["detected"] / defect_breakdown_df["count"]
defect_breakdown_df = defect_breakdown_df.sort_values(["recall", "count", "defect_type"], ascending=[True, False, True]).reset_index(drop=True)

display(defect_breakdown_df)
output_path = local_evaluation_dir / "selected_defect_breakdown.csv"
defect_breakdown_df.to_csv(output_path, index=False)
print(f"Saved defect breakdown to {output_path}")

